In [26]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.decomposition import PCA

# ML Models
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier, BaggingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    matthews_corrcoef, cohen_kappa_score
)

import warnings
warnings.filterwarnings("ignore")

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


In [27]:
df = pd.read_csv("GamingStudy_data.csv")
print(f"Shape: {df.shape}")
df.head(10)

Shape: (13464, 55)


,S. No.,Timestamp,GAD1,GAD2,GAD3,GAD4,GAD5,GAD6,GAD7,GADE,...,Birthplace,Residence,Reference,Playstyle,accept,GAD_T,SWL_T,SPIN_T,Residence_ISO3,Birthplace_ISO3
0,1,42052.00437,0,0,0,0,1,0,0,Not difficult at all,...,USA,USA,Reddit,Singleplayer,Accept,1,23,5.0,USA,USA
1,2,42052.00680,1,2,2,2,0,1,0,Somewhat difficult,...,USA,USA,Reddit,Multiplayer - online - with strangers,Accept,8,16,33.0,USA,USA
2,3,42052.03860,0,2,2,0,0,3,1,Not difficult at all,...,Germany,Germany,Reddit,Singleplayer,Accept,8,17,31.0,DEU,DEU
3,4,42052.06804,0,0,0,0,0,0,0,Not difficult at all,...,USA,USA,Reddit,Multiplayer - online - with online acquaintanc...,Accept,0,17,11.0,USA,USA
4,5,42052.08948,2,1,2,2,2,3,2,Very difficult,...,USA,South Korea,Reddit,Multiplayer - online - with strangers,Accept,14,14,13.0,KOR,USA
5,6,42052.13119,0,0,0,0,0,1,0,Not difficult at all,...,USA,USA,Reddit,Multiplayer - online - with real life friends,Accept,1,17,13.0,USA,USA
6,7,42052.14622,0,0,0,0,0,0,0,Not difficult at all,...,USA,USA,Reddit,Multiplayer - online - with online acquaintanc...,Accept,0,16,26.0,USA,USA
7,8,42052.15930,0,0,0,0,0,0,0,NaN,...,USA,Japan,Reddit,Singleplayer,Accept,0,23,NaN,JPN,USA
8,9,42052.19737,2,3,2,2,0,1,2,Very difficult,...,USA,USA,Reddit,Multiplayer - online - with strangers,Accept,12,12,55.0,USA,USA
9,10,42052.22995,2,1,2,2,2,1,0,Somewhat difficult,...,Finland,Finland,Reddit,Multiplayer - online - with online acquaintanc...,Accept,10,13,26.0,FIN,FIN


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13464 entries, 0 to 13463
Data columns (total 55 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   S. No.           13464 non-null  int64  
 1   Timestamp        13464 non-null  float64
 2   GAD1             13464 non-null  int64  
 3   GAD2             13464 non-null  int64  
 4   GAD3             13464 non-null  int64  
 5   GAD4             13464 non-null  int64  
 6   GAD5             13464 non-null  int64  
 7   GAD6             13464 non-null  int64  
 8   GAD7             13464 non-null  int64  
 9   GADE             12815 non-null  str    
 10  SWL1             13464 non-null  int64  
 11  SWL2             13464 non-null  int64  
 12  SWL3             13464 non-null  int64  
 13  SWL4             13464 non-null  int64  
 14  SWL5             13464 non-null  int64  
 15  Game             13464 non-null  str    
 16  Platform         13464 non-null  str    
 17  Hours            13434 

In [29]:
df.describe()

,S. No.,Timestamp,GAD1,GAD2,GAD3,GAD4,GAD5,GAD6,GAD7,SWL1,...,SPIN13,SPIN14,SPIN15,SPIN16,SPIN17,Narcissism,Age,GAD_T,SWL_T,SPIN_T
count,13464.000000,13464.000000,13464.000000,13464.000000,13464.000000,13464.000000,13464.000000,13464.000000,13464.000000,13464.000000,...,13277.000000,13308.000000,13317.000000,13317.000000,13289.000000,13441.000000,13464.000000,13464.000000,13464.000000,12814.000000
mean,7096.839201,42054.841222,0.860963,0.673351,0.965761,0.724079,0.488042,0.911022,0.588755,3.720440,...,0.538827,1.252405,1.411054,0.620635,0.935962,2.027677,20.930407,5.211973,19.788844,19.848525
std,4114.478220,0.272948,0.926542,0.915724,0.982776,0.921971,0.837014,0.931168,0.894408,1.736264,...,0.944180,1.207463,1.349874,0.961853,1.180456,1.061842,3.300897,4.713267,7.229243,13.467493
min,1.000000,42052.004370,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,18.000000,0.000000,5.000000,0.000000
25%,3532.750000,42054.716548,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,18.000000,2.000000,14.000000,9.000000
50%,7087.500000,42054.800675,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,4.000000,...,0.000000,1.000000,1.000000,0.000000,0.000000,2.000000,20.000000,4.000000,20.000000,17.000000
75%,10654.250000,42054.932112,1.000000,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,5.000000,...,1.000000,2.000000,2.000000,1.000000,2.000000,3.000000,22.000000,8.000000,26.000000,28.000000
max,14250.000000,42058.363750,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,7.000000,...,4.000000,4.000000,4.000000,4.000000,4.000000,5.000000,63.000000,21.000000,35.000000,68.000000


In [30]:
cols_to_drop = [
    # Raw item-level scores (totals are retained)
    "GAD1","GAD2","GAD3","GAD4","GAD5","GAD6","GAD7",
    "SWL1","SWL2","SWL3","SWL4","SWL5",
    "SPIN1","SPIN2","SPIN3","SPIN4","SPIN5","SPIN6","SPIN7",
    "SPIN8","SPIN9","SPIN10","SPIN11","SPIN12","SPIN13","SPIN14","SPIN15",
    "SPIN16","SPIN17",
    # Redundant / admin columns
    "Birthplace_ISO3", "Birthplace", "Residence",
    "Reference", "accept", "highestleague", "League",
    "Degree", "whyplay", "earnings", "streams",
]

df.columns = df.columns.str.strip()
df = df.drop(columns=cols_to_drop, errors="ignore")
print(f"Remaining columns ({len(df.columns)}): {list(df.columns)}")
df.head()

Remaining columns (15): ['S. No.', 'Timestamp', 'GADE', 'Game', 'Platform', 'Hours', 'Narcissism', 'Gender', 'Age', 'Work', 'Playstyle', 'GAD_T', 'SWL_T', 'SPIN_T', 'Residence_ISO3']


,S. No.,Timestamp,GADE,Game,Platform,Hours,Narcissism,Gender,Age,Work,Playstyle,GAD_T,SWL_T,SPIN_T,Residence_ISO3
0,1,42052.00437,Not difficult at all,Skyrim,"Console (PS, Xbox, ...)",15.0,1.0,Male,25,Unemployed / between jobs,Singleplayer,1,23,5.0,USA
1,2,42052.00680,Somewhat difficult,Other,PC,8.0,1.0,Male,41,Unemployed / between jobs,Multiplayer - online - with strangers,8,16,33.0,USA
2,3,42052.03860,Not difficult at all,Other,PC,0.0,4.0,Female,32,Employed,Singleplayer,8,17,31.0,DEU
3,4,42052.06804,Not difficult at all,Other,PC,20.0,2.0,Male,28,Employed,Multiplayer - online - with online acquaintanc...,0,17,11.0,USA
4,5,42052.08948,Very difficult,Other,"Console (PS, Xbox, ...)",20.0,1.0,Male,19,Employed,Multiplayer - online - with strangers,14,14,13.0,KOR


In [31]:
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100,
})
print(missing_data.to_string())

fig = px.bar(
    missing_data, x="Column", y="Missing_Percentage",
    title="Missing Values Percentage by Column",
    labels={"Missing_Percentage": "Missing %"},
    color="Missing_Percentage",
    color_continuous_scale="Reds"
)
fig.show()

                        Column  Missing_Count  Missing_Percentage
S. No.                  S. No.              0            0.000000
Timestamp            Timestamp              0            0.000000
GADE                      GADE            649            4.820261
Game                      Game              0            0.000000
Platform              Platform              0            0.000000
Hours                    Hours             30            0.222816
Narcissism          Narcissism             23            0.170826
Gender                  Gender              0            0.000000
Age                        Age              0            0.000000
Work                      Work             38            0.282234
Playstyle            Playstyle              0            0.000000
GAD_T                    GAD_T              0            0.000000
SWL_T                    SWL_T              0            0.000000
SPIN_T                  SPIN_T            650            4.827689
Residence_

In [32]:
df = df.dropna()
print(f"Shape after dropping NaN rows: {df.shape}")
df.head()

Shape after dropping NaN rows: (12026, 15)


,S. No.,Timestamp,GADE,Game,Platform,Hours,Narcissism,Gender,Age,Work,Playstyle,GAD_T,SWL_T,SPIN_T,Residence_ISO3
0,1,42052.00437,Not difficult at all,Skyrim,"Console (PS, Xbox, ...)",15.0,1.0,Male,25,Unemployed / between jobs,Singleplayer,1,23,5.0,USA
1,2,42052.00680,Somewhat difficult,Other,PC,8.0,1.0,Male,41,Unemployed / between jobs,Multiplayer - online - with strangers,8,16,33.0,USA
2,3,42052.03860,Not difficult at all,Other,PC,0.0,4.0,Female,32,Employed,Singleplayer,8,17,31.0,DEU
3,4,42052.06804,Not difficult at all,Other,PC,20.0,2.0,Male,28,Employed,Multiplayer - online - with online acquaintanc...,0,17,11.0,USA
4,5,42052.08948,Very difficult,Other,"Console (PS, Xbox, ...)",20.0,1.0,Male,19,Employed,Multiplayer - online - with strangers,14,14,13.0,KOR


In [33]:
gade_map = {
    "Not difficult at all": 0,
    "Somewhat difficult":   1,
    "Moderately difficult": 2,
    "Very difficult":       3,
    "Extremely difficult":  4,
}
df["GADE_enc"] = df["GADE"].map(gade_map)
df = df.drop(columns=["GADE"])
print("GADE_enc value counts:")
print(df["GADE_enc"].value_counts().sort_index())

GADE_enc value counts:
GADE_enc
0    5855
1    4827
3     962
4     382
Name: count, dtype: int64


In [34]:
text = df["Playstyle"].str.lower().fillna("")

df["ps_singleplayer"]              = text.str.contains(r"single|solo|alone", regex=True).astype(int)
df["ps_multiplayer_irl"]           = text.str.contains(r"real life|irl|girlfriend|boyfriend|partner|family|room", regex=True).astype(int)
df["ps_multiplayer_online_friends"]= text.str.contains(r"online friend|online friends|acquaintance|teammate|internet", regex=True).astype(int)
df["ps_multiplayer_strangers"]     = text.str.contains(r"stranger|random|soloq|solo q|ranked", regex=True).astype(int)

df = df.drop(columns=["Playstyle"])

In [35]:
top_games = df["Game"].value_counts().nlargest(5).index
df["Game_simple"] = df["Game"].where(df["Game"].isin(top_games), "Other")
df = pd.get_dummies(df, columns=["Game_simple"], drop_first=True)
df = df.drop(columns=["Game"])

In [36]:
europe        = ["DEU","FRA","GBR","ITA","ESP","NLD","SWE","NOR","FIN"]
north_america = ["USA","CAN"]
asia          = ["JPN","KOR","CHN","TWN"]

def region(code):
    if code in europe:        return "Europe"
    if code in north_america: return "NorthAmerica"
    if code in asia:          return "Asia"
    return "Other"

df["Region"] = df["Residence_ISO3"].apply(region)
df = pd.get_dummies(df, columns=["Region"], drop_first=True)
df = df.drop(columns=["Residence_ISO3"])

In [37]:
df = pd.get_dummies(df, columns=["Gender", "Platform", "Work"], drop_first=True)
# Cast bool dummies to int for model compatibility
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

In [38]:
df = df.drop(columns=["S. No.", "Timestamp"], errors="ignore")
print(f"Final encoded shape: {df.shape}")
df.head()

Final encoded shape: (12026, 25)


,Hours,Narcissism,Age,GAD_T,SWL_T,SPIN_T,GADE_enc,ps_singleplayer,ps_multiplayer_irl,ps_multiplayer_online_friends,...,Region_Europe,Region_NorthAmerica,Region_Other,Gender_Male,Gender_Other,Platform_PC,Platform_Smartphone / Tablet,Work_Student at college / university,Work_Student at school,Work_Unemployed / between jobs
0,15.0,1.0,25,1,23,5.0,0,1,0,0,...,0,1,0,1,0,0,0,0,0,1
1,8.0,1.0,41,8,16,33.0,1,0,0,0,...,0,1,0,1,0,1,0,0,0,1
2,0.0,4.0,32,8,17,31.0,0,1,0,0,...,1,0,0,0,0,1,0,0,0,0
3,20.0,2.0,28,0,17,11.0,0,0,0,1,...,0,1,0,1,0,1,0,0,0,0
4,20.0,1.0,19,14,14,13.0,3,0,0,0,...,0,0,0,1,0,0,0,0,0,0


In [39]:
# Bin GAD_T into severity levels
# 0 = Minimal (0-4), 1 = Mild (5-9), 2 = Moderate (10-14), 3 = Severe (15-21)
df["GAD_class"] = pd.cut(
    df["GAD_T"],
    bins=[-1, 4, 9, 14, 21],
    labels=[0, 1, 2, 3]
).astype(int)

print("Class distribution:")
print(df["GAD_class"].value_counts().sort_index())

Class distribution:
GAD_class
0    6404
1    3440
2    1507
3     675
Name: count, dtype: int64


In [40]:
# Columns that are targets or would leak the label
y_cols = ["GAD_T", "GAD_class", "SWL_T", "SPIN_T"]

# Input feature prefixes
input_prefixes = ["Gender_", "Region_", "Residence_", "Work_", "Game_", "Platform_", "ps_"]
numeric_inputs = ["Age", "Hours", "Narcissism", "GADE_enc"]

prefix_cols = [col for col in df.columns if any(col.startswith(p) for p in input_prefixes)]
X_cols = [c for c in (numeric_inputs + prefix_cols) if c not in y_cols]

print(f"Number of features: {len(X_cols)}")
print(X_cols)

Number of features: 22
['Age', 'Hours', 'Narcissism', 'GADE_enc', 'ps_singleplayer', 'ps_multiplayer_irl', 'ps_multiplayer_online_friends', 'ps_multiplayer_strangers', 'Game_simple_League of Legends', 'Game_simple_Other', 'Game_simple_Starcraft 2', 'Game_simple_World of Warcraft', 'Region_Europe', 'Region_NorthAmerica', 'Region_Other', 'Gender_Male', 'Gender_Other', 'Platform_PC', 'Platform_Smartphone / Tablet', 'Work_Student at college / university', 'Work_Student at school', 'Work_Unemployed / between jobs']


In [41]:
X = df[X_cols].copy()
y = df["GAD_class"].copy()

# Drop rows where target or any feature is NaN
data = pd.concat([X, y], axis=1).dropna()
X = data[X_cols]
y = data["GAD_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=100, stratify=y
)

print(f"Train size : {len(X_train):,}")
print(f"Test  size : {len(X_test):,}")
print(f"Class balance (train):\n{y_train.value_counts().sort_index()}")

Train size : 9,620
Test  size : 2,406
Class balance (train):
GAD_class
0    5123
1    2752
2    1205
3     540
Name: count, dtype: int64


In [42]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [43]:
y_pred = rf.predict(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall    = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1        = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print()
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy  : 0.5528
Precision : 0.5258
Recall    : 0.5528
F1 Score  : 0.5361

              precision    recall  f1-score   support

           0       0.68      0.78      0.73      1281
           1       0.39      0.35      0.37       688
           2       0.24      0.18      0.20       302
           3       0.40      0.28      0.33       135

    accuracy                           0.55      2406
   macro avg       0.43      0.40      0.41      2406
weighted avg       0.53      0.55      0.54      2406



In [44]:
cm = confusion_matrix(y_test, y_pred)
fig = px.imshow(
    cm,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=["Minimal", "Mild", "Moderate", "Severe"],
    y=["Minimal", "Mild", "Moderate", "Severe"],
    title="Confusion Matrix — Random Forest",
    color_continuous_scale="Blues",
    text_auto=True
)
fig.show()

In [45]:
importances = pd.Series(rf.feature_importances_, index=X_cols)
top15 = importances.nlargest(15).sort_values()

fig = px.bar(
    x=top15.values, y=top15.index,
    orientation="h",
    title="Top-15 Feature Importances",
    labels={"x": "Importance", "y": "Feature"}
)
fig.show()

In [46]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

# Compute per-sample weights so XGBoost corrects for class imbalance
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=False
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [47]:
y_pred_xgb = xgb.predict(X_test)

xgb_accuracy  = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb, average="weighted", zero_division=0)
xgb_recall    = recall_score(y_test, y_pred_xgb, average="weighted", zero_division=0)
xgb_f1        = f1_score(y_test, y_pred_xgb, average="weighted", zero_division=0)

print(f"Accuracy  : {xgb_accuracy:.4f}")
print(f"Precision : {xgb_precision:.4f}")
print(f"Recall    : {xgb_recall:.4f}")
print(f"F1 Score  : {xgb_f1:.4f}")
print()
print(classification_report(y_test, y_pred_xgb, zero_division=0))

Accuracy  : 0.5586
Precision : 0.5940
Recall    : 0.5586
F1 Score  : 0.5732

              precision    recall  f1-score   support

           0       0.80      0.70      0.75      1281
           1       0.43      0.42      0.43       688
           2       0.23      0.30      0.26       302
           3       0.30      0.44      0.36       135

    accuracy                           0.56      2406
   macro avg       0.44      0.47      0.45      2406
weighted avg       0.59      0.56      0.57      2406



In [48]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
fig = px.imshow(
    cm_xgb,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=["Minimal", "Mild", "Moderate", "Severe"],
    y=["Minimal", "Mild", "Moderate", "Severe"],
    title="Confusion Matrix — XGBoost",
    color_continuous_scale="Greens",
    text_auto=True
)
fig.show()

##  Side-by-Side Comparison

In [49]:
# ---------- metrics table ----------
results = pd.DataFrame({
    "Model":     ["Random Forest", "XGBoost"],
    "Accuracy":  [accuracy,      xgb_accuracy],
    "Precision": [precision,     xgb_precision],
    "Recall":    [recall,        xgb_recall],
    "F1":        [f1,            xgb_f1],
})
results = results.set_index("Model").round(4)
print(results.to_string())
results

               Accuracy  Precision  Recall      F1
Model                                             
Random Forest    0.5528     0.5258  0.5528  0.5361
XGBoost          0.5586     0.5940  0.5586  0.5732


,Accuracy,Precision,Recall,F1
Model,,,,
Random Forest,0.5528,0.5258,0.5528,0.5361
XGBoost,0.5586,0.5940,0.5586,0.5732


In [50]:
# ---------- grouped bar chart ----------
results_long = results.reset_index().melt(
    id_vars="Model", var_name="Metric", value_name="Score"
)

fig = px.bar(
    results_long,
    x="Metric", y="Score", color="Model",
    barmode="group",
    title="Random Forest vs XGBoost — Weighted Metrics",
    text_auto=".3f",
    color_discrete_map={"Random Forest": "#636EFA", "XGBoost": "#00CC96"},
    range_y=[0, 1]
)
fig.update_traces(textposition="outside")
fig.show()

In [ ]:
# ---------- feature importances side-by-side ----------
rf_imp  = pd.Series(rf.feature_importances_,  index=X_cols).nlargest(15).sort_values()
xgb_imp = pd.Series(xgb.feature_importances_, index=X_cols).nlargest(15).sort_values()

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Random Forest — Top 15", "XGBoost — Top 15"),
    shared_xaxes=False
)
fig.add_trace(
    go.Bar(x=rf_imp.values, y=rf_imp.index, orientation="h",
           marker_color="#636EFA", name="RF"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=xgb_imp.values, y=xgb_imp.index, orientation="h",
           marker_color="#00CC96", name="XGBoost"),
    row=1, col=2
)
fig.update_layout(height=500, title_text="Feature Importances Comparison", showlegend=False)
fig.show()